# Analysis of Carcará-X's devices scans

The following cells facilitate the analysis of scans obtained and stored in `.hdf5` files. 
It mainly uses methods implemented in the `scananalysis` module, using the `Histogram2DAnalyzer` class 
to extract beam image properties.

In [ ]:
# Import necessary modules and functions
import caxscripts.scananalysis as _sa
import numpy as np

### Main entry for data reading

Specify the working directory and file pattern for loading data,
each file representes a scan, and the pattern should match the naming 
convention of the files. A `dataset` variable is created to hold the 
data for multiple passes of the same kind of scan.

In [ ]:
# workdir = "/ibira/lnls/beamlines/carcara/Carcara-X/data"
# workdir = "/home/arnaldo.filho/Carcara/measurement/2026-03-19"
workdir = "/home/gabriel.amici/testing_data/2026-03-19/"
file_pattern = r"mirror_tx_pass[01][0-9]*"


def main(workdir, file_pattern):
    """Main function to load data and generate plots."""
    files   = _sa.files_in_directory(workdir, file_pattern)
    dataset = _sa.dataset_from_h5_files(files)
    print(f" Loaded {len(files)} files matching pattern "
          f"'{file_pattern}' \n from '{workdir}'\n Files:")

    for f in files:
        print(f" * {f}")
    
    print("\n")
    print("="*50)

    global number_of_passes
    number_of_passes = len(dataset)
    print(f"Number of passes: {number_of_passes} \t -> `number_of_passes`")

    # Usually all scans have the same number of steps, 
    # but we can check if that's the case
    global steps_per_scan
    steps_per_scan = [len(dataset[f"{i+1:02d}"]) for i in range(len(dataset))]
    if len(set(steps_per_scan)) == 1:
        steps_per_scan = steps_per_scan[0]
        print(f"Steps per scan: {steps_per_scan} \t -> `steps_per_scan`")
    else:
        print("Warning: Not all scans have the same number of steps.")
        print(f"Steps per scan: {steps_per_scan}")
    return dataset


if __name__ == "__main__":

    dataset = main(workdir, file_pattern)

### Observable plot for all passes

Main entry for analysis. Add observable name to `observables` and adjust 
`pass_interval` and `step_interval`, to control what's shown.

The following cell plots many observables across every pass and step:


In [ ]:
# Currently supported observables are:
#  - Any property present in the .hdf5 files:
#    - tx, cs_ty, cs_rx, ry, cs_rz, y1, y2, y3, photocollector, etc.
#  - Derived properties:
#    - centroid, fwhm, intensity, etc.

observables = [
    'centroid',
    'fwhm',
    'intensity']

pass_interval = (0, number_of_passes)
step_interval = (0, steps_per_scan)

# Chose set interval of scans to plot.
first_pass, last_pass = pass_interval
# Choose set interval of steps to plot.
sel_data = dict(list(dataset.items())[first_pass:last_pass+1])

fr, upto = step_interval

_sa.scan_plot(sel_data, observables, first_item=fr, last_item=upto)

#### Compare centroids at the beginning, middle and end points of each pass.

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np

cty_0 = []
cty_m = []
cty_f = []
npti = 0
nptf = 12

for data in dataset.values():
    bc  = _sa.beam_centroid(data, 'mirror.tx')
    tx0 = bc[1][npti]
    cy0 = bc[2][npti]
    cty_0.append(np.array([tx0, cy0[1]]))

    txm = bc[1][(npti + nptf) // 2]
    cym = bc[2][(npti + nptf) // 2]
    cty_m.append(np.array([txm, cym[1]]))

    txf = bc[1][nptf]
    cyf = bc[2][nptf]
    cty_f.append(np.array([txf, cyf[1]]))

cty_0 = np.array(cty_0)
cty_m = np.array(cty_m)
cty_f = np.array(cty_f)

n = len(cty_0)
colors = cm.viridis(np.linspace(0, 1, n))

fig, (axx, axc, axy) = plt.subplots(1, 3, figsize=(24, 6))

for i, (idx, cy) in enumerate(cty_0):
    axx.scatter(idx, cy, color=colors[i], zorder=3, s=60)
    axx.annotate(str(i), (idx, cy), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)

for i, (idx, cy) in enumerate(cty_m):
    axc.scatter(idx, cy, color=colors[i], zorder=3, s=60)
    axc.annotate(str(i), (idx, cy), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)

for i, (idx, cy) in enumerate(cty_f):
    axy.scatter(idx, cy, color=colors[i], zorder=3, s=60)
    axy.annotate(str(i), (idx, cy), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)


sm = plt.cm.ScalarMappable(cmap='viridis',
                            norm=plt.Normalize(vmin=0, vmax=n - 1))
plt.colorbar(sm, ax=axx, label='Sequence index')
plt.colorbar(sm, ax=axc, label='Sequence index')
plt.colorbar(sm, ax=axy, label='Sequence index')

xmin, xmax = np.min(cty_0[:, 0]) * 1.002, np.max(cty_0[:, 0]) * 0.998
ymin, ymax = np.min(cty_0[:, 1]) * 0.998, np.max(cty_0[:, 1]) * 1.002

axx.grid(True)
axx.set_title("Centroid Y vs. Tx, Initial position")
axx.set_xlabel('Tx initial (mm)')
axx.set_ylabel('Centroid Y (px)')
axx.set_xlim(xmin, xmax)
axx.set_ylim(ymin, ymax)

axc.grid(True)
axc.set_title("Centroid Y vs. Tx, Intermediate position")
axc.set_xlabel('Tx intermediate (mm)')
axc.set_ylabel('Centroid Y (px)')
axc.set_xlim(xmin, xmax)
axc.set_ylim(ymin, ymax)
axc.grid(True)

axy.grid(True)
axy.set_title("Centroid Y vs. Tx, Final position")
axy.set_xlabel('Tx final (mm)')
axy.set_ylabel('Centroid Y (px)')
axy.set_xlim(xmin, xmax)
axy.set_ylim(ymin, ymax)

plt.show()

#### Mean and median of a single curve, for testing.

In [ ]:
observable = 'fwhm'
stats = _sa.observable_statistics(dataset, observable)
print(f" observable = '{observable}'")
for key, value in stats.items():
    print(f">>> {key}: {value}  \t ({value.shape})\n")


In [ ]:
idx = stats['xval']
cy = stats['mean'][0]
#  cy = stats['mean'][1]
dy = stats['std_dev'][0]
# dy = stats['std_dev'][1]
my = stats['median'][0]
# my = stats['median'][1]

fig, (axmx, axmy) = plt.subplots(1, 2, figsize=(15, 6))

print(f" idx: {idx.shape},\n"
      f" yv: {cy.shape},"
      # f" cy: {cy.shape},\n"
      # f" dx: {dx.shape},"
      f" dy: {dy.shape},\n"
      # f" mx: {mx.shape},"
      f" my: {my.shape}")

axmx.errorbar(idx, cy, yerr=dy, fmt='o-', label=f'{observable} mean')
axmx.set_ylabel(f'x {observable} Mean')
axmx.set_title(f'Tx X {observable} Mean')

axmy.set_ylabel(f'x {observable} Median')
axmy.set_title(f'Tx X {observable} Median')
axmy.plot(idx, my, 's-', label=f'{observable} Median', color='red')

for ax in [axmx, axmy]:
    ax.grid(True)
    ax.set_xlabel('Tx')
    ax.legend()
    ax.grid(True)

### Sequence of beam images.

In [ ]:
# Working directory for saving the plots.
wdir = "/home/ABTLUS/arnaldo.filho/Beam_profiling/beam_profiles"

# Plot centroids and images for all datasets.
for scanpass, data in dataset.items():
    _sa.centroid_plot(data, scanpass, wdir=wdir, save_fmt='')


#### Evaluation of differences of centroid position along experimental passes.

In [ ]:
import numpy as np

pixsize = 0.48  # um/px

initfindiff = []
scandiff = []
for scanpass, data in dataset.items():
    print(f"\n##### Scan pass: {scanpass}")

    cent = _sa.observable_data(data, 'centroid')
    idx = cent[2][1:-1] * pixsize
    cy = cent[3][0][1:-1] * pixsize

    ay, by = np.polyfit(idx, cy, 1)
    cxfit = ay * idx + by
    c0a = cent[3][0][0] * pixsize
    c0b = cent[3][0][-1] * pixsize
    c0x = cy[len(cy) // 2]

    c0fit = ay * idx[len(idx) // 2] + by

    print("Linear fit parameters:"
          f"               a = {ay:.4f}, b = {by:.4f}")
    print("Initial and final centroid values:"
          f"   {c0a:.2f} → {c0b:.2f}")
    print("Central centroid value along scan:"
          f"   {c0x:.2f}")

    dinfin = (c0b / c0a - 1) * 100
    cdiff  = (c0x / c0a - 1) * 100
    scandiff.append(cdiff)
    initfindiff.append(dinfin)
    print("Centroid differences, relative to initial position:")
    print(f"\t at center, along scan:   {(c0x - c0a) / c0a * 100 :.2f}%")
    print(f"\t after scan:              {dinfin:.2f}%")
    print(f"\t at center, based on fit: {(c0fit - c0a) / c0a * 100 :.2f}%")

scandiff = np.array(scandiff)
initfindiff = np.array(initfindiff)

#### Correlation analysis.

In [ ]:
ifdiff = np.array(initfindiff)    # - np.mean(initfindiff)
sdiff  = np.array(scandiff)       # - np.mean(scandiff)

corr_mat = np.corrcoef(sdiff, ifdiff)
print("\nCorrelation coefficient between"
      f" initial/final change and scan change: {corr_mat[0, 1]:.4f}")

# cross_correlation = np.correlate(sdiff, ifdiff, mode='same')
cc = correlate(sdiff, ifdiff)
lags = np.arange(-len(sdiff) + 1, len(sdiff))
best_lag = lags[np.argmax(cc)]
print(f"Best correlation: {np.max(cc)}")
print(f"Shift needed: {best_lag} indices")


# print("Cross-correlation between initial/final change and scan change:"
#       f" {cross_correlation}")

fig, axs = plt.subplots(figsize=(10, 6))
sc = np.array(list(dataset.keys()))
axs.plot(sc, ifdiff, marker='o', label='initial/final change')
axs.plot(sc, sdiff, marker='s', label='scan/initial change')
# ax.plot(sc, cc, marker='^',
#         label='cross-correlation (normalized)')
axs.grid(True)
# ax.set_ylim(0, max(max(ifdiff), max(sdiff)) * 1.2)
axs.set_xlabel('Scan pass')
axs.set_ylabel('Centroid change (%)')
axs.legend(loc='upper center', bbox_to_anchor=(0.85, 0.72))
axs.set_title('Relative change in central centroid position across scan passes')

print(" Initial/final mean and std. deviation values:"
      f" {np.mean(ifdiff):.4f} ± {np.std(ifdiff):.4f}")
print(" Scan/initial mean and std. deviation values:"
      f" {np.mean(sdiff):.4f} ± {np.std(sdiff):.4f}")

plt.show()

In [ ]:
fig, axs = plt.subplots(figsize=(10, 6))
axs.plot(cc, marker='o', label='initial/final change')
axs.grid(True)
axs.set_title("Cross-correlation between initial/final change and scan change")
axs.set_xlabel('Lag')
axs.set_ylabel('Correlation coefficient')
# ax.legend()
plt.show()
print(np.max(cc))

#### Linear fitting to centroid positions along a scanning.

In [ ]:
import matplotlib.pyplot as plt

data = dataset['01']
motor, scans, xvals, centrs, sigmas = observable_data(data, 'centroid')
idx  = xvals[1:-1]

cx   = centrs[0] * pixsize
sx   = sigmas[0] * pixsize

cy   = centrs[1] * pixsize
sy   = sigmas[1] * pixsize

wx = 1 / sx[1:-1]
ax, bx = np.polyfit(idx, cx[1:-1], 1, w=wx)
cxfit = ax * idx + bx

wy = 1 / sy[1:-1]
ay, by = np.polyfit(idx, cy[1:-1], 1, w=wy)
cyfit = ay * idx + by

cx0fit = ax * idx[len(idx) // 2] + bx
cy0fit = ay * idx[len(idx) // 2] + by

fig, (axx, axy) = plt.subplots(1, 2, figsize=(14, 6))

axx.errorbar(idx, cx[1:-1], yerr=sx[1:-1], fmt='o-', label='Centroid X')
axx.plot(idx, cxfit, 'r--', label='Linear Fit')
axx.plot(xvals[0], cx[0], 'gs', label='Init. Centr. X')
axx.plot(xvals[-1], cx[-1], 'rs', label='Fin. Centr. X')

axy.errorbar(idx, cy[1:-1], yerr=sy[1:-1], fmt='o-', label='Centroid Y')
axy.plot(idx, cyfit, 'r--', label='Linear Fit')
axy.plot(xvals[0], cy[0], 'gs', label='Init. Centr. Y')
axy.plot(xvals[-1], cy[-1], 'rs', label='Fin. Centr. Y')

axx.set_xlabel(r'$Tx$ [mm]')
axx.set_ylabel(r'centroid X [$\mu$m]')
axx.set_title(r'Centroids vs $Tx$ with linear fit')
axx.legend()
axx.grid(True)

axy.set_xlabel(r'$Tx$ [mm]')
axy.set_ylabel(r'centroid Y [$\mu$m]')
axy.set_title(r'Centroids vs $Tx$ with linear fit')
axy.legend()
axy.grid(True)

phi = np.arctan(ay/1000)
print(f"Rotation angle from linear fit: {phi * 1e3:.4f} urad")

#### Test with a specific scan frame.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dx = 4
threshold = 10
data = dataset['03']
scandata = '0004'

img = data[f'scan-{scandata}']['dvf_B1']['data']
centroid = beam_centroid(data, 'mirror.tx')
idx, (cy, cy) = centroid[int(scandata)]

roi_avg = np.mean(img[cy-dx:cy+dx, cy-dx:cy+dx])
mean = np.mean(img)
ratio_rtom = roi_avg / mean

print(f"ROI average: {roi_avg:.2f}, Mean: {mean:.2f},"
      f" Ratio (ROI/Mean): {ratio_rtom:.2f}")

print(idx, cy, cy)
plt.imshow(img)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

data = dataset['03']
scan = data['scan-0000']
img1 = scan['dvf_A1']['data']
img2 = scan['dvf_B1']['data']

cy = np.sum(img2, axis=0).argmax()
cy = np.sum(img2, axis=1).argmax()

ax1.imshow(img1, cmap='viridis')
ax1.set_title('dvf_A1')

ax2.imshow(img2, cmap='viridis')
ax2.set_title('dvf_B1')

print(f"Centroid (dvf_B1): {cy}, {cy}")